# Invariance — invariance of a uniform magnetic field — 3D

`MagneticStandalone3D` · ASN 28 validation case (TRUST-LiMe).

## Introduction

Validation of the **standalone magnetic (induction) equation** added by
TRUST-LiMe:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

with the magnetic diffusivity $\eta$ and the transporting velocity $u$ supplied
as data (no Navier–Stokes coupling in this first phase). Every case is
**self-contained** (inline `Mailler`/`Pave` mesh, no `.med` file) and records
point probes (`Postraitement/Sondes`) into `.son` files which are compared
below against a closed-form solution.

This notebook runs **one and only one test case**: ``invariance``. Per the TRUST
validation-form layout (`Validation/Rapports_automatiques/<Category>/<FormName>/`),
the other `MagneticStandalone3D` cases live in their own sibling folders, each
with `<FormName>/<FormName>.ipynb` + `src/<case>.data`.

## What it validates

With $b\equiv b_0$ and $u=0$ both transport terms vanish identically, so
$b(x,t)$ must stay equal to $b_0$ for all $t$. Machine-precision invariant:
it catches spurious sources, a wrong time factor or bad CL assembly.

## Contracts checked

| # | Modelling decision | Check |
|---|---|:--:|
| 5 | no spurious source (equilibria preserved) | exact invariant |


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.


In [ ]:
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    path = f"{BUILD}/{case}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def ratio_error(case, tags, ref):
    """Max |b(t)/b(0) - ref(t)| over the given probes and all recorded times."""
    worst = 0.0
    for tag in tags:
        t, b = read_son(case, tag)
        if b[0] == 0:
            continue
        worst = max(worst, float(np.max(np.abs(b / b[0] - ref(t)))))
    return worst

print("BUILD =", BUILD)


## Run

Sequential-only: the dataset is run by `trustutils` on one processor (no `partition()`, no `nbProcs`).


In [ ]:
import pathlib, shutil

from trustutils import run

run.reset()
run.addCase(".", "invariance.data")      # sequential-only: never nbProcs > 1
run.printCases()
run.runCases()

# trustutils runs the case in its build tree; the .son probes are written next
# to the dataset in src/, so copy them where read_son() looks for them.
build = pathlib.Path(run.BUILD_DIRECTORY)
build.mkdir(parents=True, exist_ok=True)
for son in pathlib.Path("src").glob("invariance_*.son"):
    shutil.copy2(son, build / son.name)


## Checks


In [ ]:
t, b = read_son("invariance", "SONDE_CBX")

b0 = 3.0
err = np.max(np.abs(b - b0)) / abs(b0)
print(f"final time       : {t[-1]:.6g}")
print(f"max |b - b0|/b0 : {err:.3e}")
assert err < 1e-10, f"uniform field not conserved (rel. error {err:.3e})"
print("OK: uniform magnetic field is invariant.")


## Computer Performance


In [ ]:
run.tablePerf()